In [1]:
import pandas as pd

In [2]:
price_dataset = pd.read_csv("../data/raw/entso-e-prices/Greece.csv")
price_dataset["datetime"] = pd.to_datetime(price_dataset["Datetime (UTC)"])

In [3]:
price_dataset.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe),datetime
0,Greece,GRC,2015-01-01 00:00:00,2015-01-01 02:00:00,48.78,2015-01-01 00:00:00
1,Greece,GRC,2015-01-01 01:00:00,2015-01-01 03:00:00,31.10,2015-01-01 01:00:00
2,Greece,GRC,2015-01-01 02:00:00,2015-01-01 04:00:00,20.78,2015-01-01 02:00:00
3,Greece,GRC,2015-01-01 03:00:00,2015-01-01 05:00:00,25.40,2015-01-01 03:00:00
4,Greece,GRC,2015-01-01 04:00:00,2015-01-01 06:00:00,26.00,2015-01-01 04:00:00


In [4]:
price_dataset = price_dataset.drop(columns=["Country", "ISO3 Code", 'Datetime (Local)', 'Datetime (UTC)'])

In [5]:
price_dataset.head()

,Price (EUR/MWhe),datetime
0,48.78,2015-01-01 00:00:00
1,31.10,2015-01-01 01:00:00
2,20.78,2015-01-01 02:00:00
3,25.40,2015-01-01 03:00:00
4,26.00,2015-01-01 04:00:00


In [6]:
solar_dataset = pd.read_csv("../data/raw/GR/solar-raw.csv")

In [8]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [9]:
solar_dataset.columns = ["datetime", "solar_generation_MW"]

In [10]:
solar_dataset.head()

,datetime,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [11]:
solar_dataset["datetime"] = pd.to_datetime(solar_dataset["datetime"], utc=True).dt.tz_localize(None)

In [15]:
solar_dataset.isna().sum()

datetime               0
solar_generation_MW    0
dtype: int64

In [16]:
meteo_dataset = pd.read_csv("../data/raw/meteo-raw-greece.csv")

In [17]:
meteo_dataset.head()

,time,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,2022-01-01T00:00,4.0,8.3,34,0,0.0,0.0
1,2022-01-01T01:00,3.0,6.9,39,0,0.0,0.0
2,2022-01-01T02:00,4.3,5.4,37,0,0.0,0.0
3,2022-01-01T03:00,6.0,3.6,37,0,0.0,0.0
4,2022-01-01T04:00,5.9,1.8,37,0,0.0,0.0


In [18]:
meteo_dataset["datetime"] = pd.to_datetime(meteo_dataset["time"])

In [19]:
meteo_dataset.columns

Index(['time', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m',
       'cloud_cover', 'shortwave_radiation', 'precipitation', 'datetime'],
      dtype='object')

In [20]:
meteo_dataset = meteo_dataset.drop(columns=["time"])

In [21]:
meteo_dataset.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,datetime
0,4.0,8.3,34,0,0.0,0.0,2022-01-01 00:00:00
1,3.0,6.9,39,0,0.0,0.0,2022-01-01 01:00:00
2,4.3,5.4,37,0,0.0,0.0,2022-01-01 02:00:00
3,6.0,3.6,37,0,0.0,0.0,2022-01-01 03:00:00
4,5.9,1.8,37,0,0.0,0.0,2022-01-01 04:00:00


In [22]:
merged = price_dataset.merge(solar_dataset, on="datetime", how="inner")
merged = merged.merge(meteo_dataset, on="datetime", how="inner")
merged = merged.sort_values("datetime").reset_index(drop=True)

print(merged.shape)
merged.head()

(35064, 9)


,Price (EUR/MWhe),datetime,solar_generation_MW,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,155.72,2022-01-01 00:00:00,0.0,4.0,8.3,34,0,0.0,0.0
1,147.09,2022-01-01 01:00:00,0.0,3.0,6.9,39,0,0.0,0.0
2,91.00,2022-01-01 02:00:00,0.0,4.3,5.4,37,0,0.0,0.0
3,104.00,2022-01-01 03:00:00,0.0,6.0,3.6,37,0,0.0,0.0
4,140.60,2022-01-01 04:00:00,0.0,5.9,1.8,37,0,0.0,0.0


In [23]:
merged.to_csv("../data/processed/greece_merged.csv", index=False)